In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit

SILVER_TABLE = "silver_merge_demo"

source_df = (
    spark.table("silver_weather")
    .filter(col("city") == "Berlin")
    .filter(
        (col("timestamp") >= "1995-01-01")
        & (col("timestamp") < "1995-01-02")
    )
)

assert source_df.count() == 24

(
    source_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

silver_target = DeltaTable.forName(spark, SILVER_TABLE)

print("Test Silver rows:", spark.table(SILVER_TABLE).count())

Test Silver rows: 24


In [0]:
MEASURE_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

def merge_into_silver(source_df):
    
    staged_df = source_df.dropDuplicates(["city", "timestamp"])

    change_condition = " OR ".join(
        [
            f"NOT (target.{c} <=> source.{c})"
            for c in MEASURE_COLUMNS
        ]
    )

    (
        DeltaTable.forName(spark, SILVER_TABLE)
        .alias("target")
        .merge(
            staged_df.alias("source"),
            "target.city = source.city "
            "AND target.timestamp = source.timestamp"
        )
        .whenMatchedUpdateAll(condition=change_condition)
        .whenNotMatchedInsertAll()
        .execute()
    )

    history = (
        DeltaTable.forName(spark, SILVER_TABLE)
        .history(1)
        .select("operationMetrics")
        .first()["operationMetrics"]
    )

    inserted = int(history.get("numTargetRowsInserted", 0))
    updated = int(history.get("numTargetRowsUpdated", 0))

    print("Rows inserted:", inserted)
    print("Rows updated:", updated)

    return inserted, updated


In [0]:
test_source_df = spark.table(SILVER_TABLE)

before = spark.table(SILVER_TABLE).count()

inserted_1, updated_1 = merge_into_silver(test_source_df)

after_first = spark.table(SILVER_TABLE).count()

inserted_2, updated_2 = merge_into_silver(test_source_df)

after_second = spark.table(SILVER_TABLE).count()

duplicates = (
    spark.table(SILVER_TABLE).count()
    - spark.table(SILVER_TABLE)
        .select("city", "timestamp")
        .distinct()
        .count()
)

print("Before:", before)
print("After first MERGE:", after_first)
print("After second MERGE:", after_second)
print("Duplicate keys:", duplicates)

assert before == after_first == after_second == 24
assert inserted_1 == 0
assert inserted_2 == 0
assert updated_1 == 0
assert updated_2 == 0
assert duplicates == 0

print("IDEMPOTENCY TEST PASSED")

Rows inserted: 0
Rows updated: 0
Rows inserted: 0
Rows updated: 0
Before: 24
After first MERGE: 24
After second MERGE: 24
Duplicate keys: 0
IDEMPOTENCY TEST PASSED


In [0]:


LOAD_MODE = "INCREMENTAL"   


INCREMENTAL_START_DATE = "2025-12-01"
INCREMENTAL_END_DATE = "2025-12-31"


BACKFILL_CITY = "London"
BACKFILL_START_DATE = "2018-01-01"
BACKFILL_END_DATE = "2018-01-31"

print("Load mode:", LOAD_MODE)
print("Incremental window:", INCREMENTAL_START_DATE, "to", INCREMENTAL_END_DATE)
print("Backfill example:", BACKFILL_CITY, BACKFILL_START_DATE, "to", BACKFILL_END_DATE)


Load mode: INCREMENTAL
Incremental window: 2025-12-01 to 2025-12-31
Backfill example: London 2018-01-01 to 2018-01-31


In [0]:

sample_row = spark.table(SILVER_TABLE).limit(1)

controlled_update_df = (
    sample_row
    .withColumn("temperature_2m", col("temperature_2m") + lit(0.1))
    .withColumn("load_timestamp", current_timestamp())
)

inserted, updated = merge_into_silver(controlled_update_df)

print("Controlled update test:")
print("Inserted:", inserted)
print("Updated:", updated)

assert inserted == 0
assert updated == 1

print("MERGE UPDATE path verified.")


Rows inserted: 0
Rows updated: 1
Controlled update test:
Inserted: 0
Updated: 1
MERGE UPDATE path verified.
